In [0]:
%sql
SELECT
  COUNT(*)                                     AS total_transactions,
  SUM(Class)                                   AS total_fraud,
  ROUND(SUM(Class)/COUNT(*)*100, 4)            AS fraud_rate_pct,
  ROUND(AVG(CASE WHEN Class = 1
            THEN Amount END), 2)               AS avg_fraud_amount,
  ROUND(MAX(CASE WHEN Class = 1
            THEN Amount END), 2)               AS max_fraud_amount,
  ROUND(MIN(CASE WHEN Class = 1
            THEN Amount END), 2)               AS min_fraud_amount
FROM fraud_db.silver_transactions;

total_transactions,total_fraud,fraud_rate_pct,avg_fraud_amount,max_fraud_amount,min_fraud_amount
281918,448,0.1589,130.78,2125.87,0.01


In [0]:
%sql
SELECT
  hour_of_day,
  COUNT(*)                               AS total_txns,
  SUM(Class)                             AS fraud_count,
  ROUND(SUM(Class)/COUNT(*)*100, 4)      AS fraud_rate_pct,
  ROUND(AVG(Amount), 2)                  AS avg_amount
FROM fraud_db.silver_transactions
GROUP BY hour_of_day
ORDER BY hour_of_day;

hour_of_day,total_txns,fraud_count,fraud_rate_pct,avg_amount
0,7578,5,0.066,61.14
1,4177,10,0.2394,63.23
2,3286,48,1.4607,70.88
3,3453,16,0.4634,52.27
4,2174,19,0.874,78.22
5,2980,11,0.3691,50.84
6,4065,8,0.1968,65.46
7,7137,23,0.3223,68.8
8,10181,9,0.0884,89.34
9,15611,15,0.0961,103.96


In [0]:
%sql
SELECT
  amount_category,
  COUNT(*)                               AS total_txns,
  SUM(Class)                             AS fraud_count,
  ROUND(SUM(Class)/COUNT(*)*100, 4)      AS fraud_rate_pct,
  ROUND(AVG(Amount), 2)                  AS avg_amount
FROM fraud_db.silver_transactions
GROUP BY amount_category
ORDER BY fraud_rate_pct DESC;

amount_category,total_txns,fraud_count,fraud_rate_pct,avg_amount
large,3064,9,0.2937,1774.36
micro,95065,213,0.2241,3.77
medium,54212,116,0.214,263.66
small,129577,110,0.0849,38.69


In [0]:
%sql
SELECT
  CASE WHEN is_night = 1
       THEN 'Night (10PM-4AM)'
       ELSE 'Day (4AM-10PM)'
  END                                    AS time_period,
  COUNT(*)                               AS total_txns,
  SUM(Class)                             AS fraud_count,
  ROUND(SUM(Class)/COUNT(*)*100, 4)      AS fraud_rate_pct,
  ROUND(AVG(Amount), 2)                  AS avg_amount
FROM fraud_db.silver_transactions
GROUP BY is_night
ORDER BY fraud_rate_pct DESC;

time_period,total_txns,fraud_count,fraud_rate_pct,avg_amount
Night (10PM-4AM),46803,121,0.2585,67.07
Day (4AM-10PM),235115,327,0.1391,93.41


In [0]:
%sql
SELECT
  ROUND(Amount, 2)                       AS amount,
  hour_of_day,
  CASE WHEN is_night = 1
       THEN 'YES' ELSE 'NO'
  END                                    AS is_night,
  ROUND(fraud_probability * 100, 2)      AS fraud_prob_pct,
  risk_level,
  actual_class                           AS was_fraud
FROM fraud_db.gold_predictions
WHERE risk_level IN ('CRITICAL', 'HIGH')
ORDER BY fraud_probability DESC
LIMIT 50;

amount,hour_of_day,is_night,fraud_prob_pct,risk_level,was_fraud
2.28,4.0,YES,99.98,CRITICAL,1
1.0,3.0,YES,99.98,CRITICAL,1
99.85,21.0,NO,99.98,CRITICAL,1
209.65,2.0,YES,99.98,CRITICAL,1
75.86,2.0,YES,99.98,CRITICAL,1
53.95,11.0,NO,99.98,CRITICAL,1
316.06,2.0,YES,99.98,CRITICAL,1
1.0,2.0,YES,99.98,CRITICAL,1
37.32,11.0,NO,99.98,CRITICAL,1
99.99,14.0,NO,99.98,CRITICAL,1


In [0]:
%sql
SELECT
  COUNT(*)                               AS total_predictions,
  SUM(CASE WHEN predicted_class = 1
       AND actual_class = 1
       THEN 1 END)                       AS true_positives,
  SUM(CASE WHEN predicted_class = 1
       AND actual_class = 0
       THEN 1 END)                       AS false_positives,
  SUM(CASE WHEN predicted_class = 0
       AND actual_class = 1
       THEN 1 END)                       AS false_negatives,
  SUM(CASE WHEN predicted_class = 0
       AND actual_class = 0
       THEN 1 END)                       AS true_negatives,
  ROUND(
    SUM(CASE WHEN predicted_class = actual_class
         THEN 1 END) / COUNT(*) * 100, 2
  )                                      AS accuracy_pct
FROM fraud_db.gold_predictions;

total_predictions,true_positives,false_positives,false_negatives,true_negatives,accuracy_pct
56384,84,133,6,56161,99.75


In [0]:
%sql
SELECT
  risk_level,
  COUNT(*)                               AS total_txns,
  SUM(actual_class)                      AS actual_fraud,
  ROUND(AVG(fraud_probability)*100, 2)   AS avg_fraud_prob_pct,
  ROUND(AVG(Amount), 2)                  AS avg_amount
FROM fraud_db.gold_predictions
GROUP BY risk_level
ORDER BY avg_fraud_prob_pct DESC;

risk_level,total_txns,actual_fraud,avg_fraud_prob_pct,avg_amount
CRITICAL,77,72,98.75,130.33
HIGH,22,7,77.11,82.58
MEDIUM,118,5,58.94,67.16
LOW,56167,6,2.33,89.21


In [0]:
# ============================================
# CELL 8 — Full pipeline verification
# ============================================

tables = [
    "fraud_db.bronze_transactions",
    "fraud_db.silver_transactions",
    "fraud_db.gold_features",
    "fraud_db.gold_hourly_summary",
    "fraud_db.gold_predictions"
]

print("=" * 55)
print("       FRAUD DETECTION PIPELINE — FINAL CHECK")
print("=" * 55)

for table in tables:
    try:
        count = spark.table(table).count()
        print(f"✅ {table:40s} {count:,} rows")
    except Exception as e:
        print(f"❌ {table:40s} ERROR: {e}")

print("=" * 55)

# Check MLflow models
import mlflow
client = mlflow.tracking.MlflowClient()

print("\n=== MLFLOW REGISTERED MODELS ===")
for mv in client.search_model_versions(""):
    print(
        f"✅ {mv.name:30s} "
        f"v{mv.version}  "
        f"Stage: {mv.current_stage}"
    )

print("\n🎉 Pipeline 100% complete!")
print("   Ready for GitHub and LinkedIn!")

       FRAUD DETECTION PIPELINE — FINAL CHECK
✅ fraud_db.bronze_transactions             284,807 rows
✅ fraud_db.silver_transactions             281,918 rows
✅ fraud_db.gold_features                   281,918 rows
✅ fraud_db.gold_hourly_summary             24 rows
✅ fraud_db.gold_predictions                56,384 rows

=== MLFLOW REGISTERED MODELS ===


---------------------------------------------------------------------------
RestException                             Traceback (most recent call last)
File <command-6065816812646344>, line 31
     28 client = mlflow.tracking.MlflowClient()
     30 print("\n=== MLFLOW REGISTERED MODELS ===")
---> 31 for mv in client.search_model_versions(""):
     32     print(
     33         f"✅ {mv.name:30s} "
     34         f"v{mv.version}  "
     35         f"Stage: {mv.current_stage}"
     36     )
     38 print("\n🎉 Pipeline 100% complete!")

File /databricks/python/lib/python3.12/site-packages/mlflow/tracking/client.py:4734, in MlflowClient.search_model_versions(self, filter_string, max_results, order_by, page_token)
   4654 def search_model_versions(
   4655     self,
   4656     filter_string: Optional[str] = None,
   (...)
   4659     page_token: Optional[str] = None,
   4660 ) -> PagedList[ModelVersion]:
   4661     """
   4662     Search for model versions in backend that satisfy the filt

In [0]:
# ============================================
# CELL 8 — Full pipeline verification
# (Fixed: MLflow model search)
# ============================================

tables = [
    "fraud_db.bronze_transactions",
    "fraud_db.silver_transactions",
    "fraud_db.gold_features",
    "fraud_db.gold_hourly_summary",
    "fraud_db.gold_predictions"
]

print("=" * 55)
print("   FRAUD DETECTION PIPELINE — FINAL CHECK")
print("=" * 55)

# ── Check all Delta tables ────────────────────────────────────────
print("\n=== DELTA TABLES ===")
for table in tables:
    try:
        count = spark.table(table).count()
        print(f"✅ {table:45s} {count:,} rows")
    except Exception as e:
        print(f"❌ {table:45s} ERROR: {e}")

# ── Check MLflow registered models ───────────────────────────────
print("\n=== MLFLOW REGISTERED MODELS ===")
import mlflow
client = mlflow.tracking.MlflowClient()

model_names = [
    "fraud_RandomForest",
    "fraud_XGBoost",
    "fraud_LogisticRegression"
]

for name in model_names:
    try:
        versions = client.search_model_versions(
            f"name='{name}'"
        )
        for mv in versions:
            print(
                f"✅ {mv.name:35s} "
                f"v{mv.version}  "
                f"Stage: {mv.current_stage}"
            )
    except Exception as e:
        print(f"❌ {name:35s} ERROR: {e}")

# ── Final summary ─────────────────────────────────────────────────
print("\n=== PIPELINE SUMMARY ===")
total_txns  = spark.table(
    "fraud_db.silver_transactions"
).count()
total_fraud = spark.sql(
    "SELECT SUM(Class) as cnt "
    "FROM fraud_db.silver_transactions"
).collect()[0]["cnt"]
critical = spark.sql(
    "SELECT COUNT(*) as cnt "
    "FROM fraud_db.gold_predictions "
    "WHERE risk_level = 'CRITICAL'"
).collect()[0]["cnt"]
high = spark.sql(
    "SELECT COUNT(*) as cnt "
    "FROM fraud_db.gold_predictions "
    "WHERE risk_level = 'HIGH'"
).collect()[0]["cnt"]

print(f"   Total transactions   : {total_txns:,}")
print(f"   Total fraud cases    : {total_fraud:,}")
print(f"   Fraud rate           : {total_fraud/total_txns*100:.4f}%")
print(f"   Critical risk txns   : {critical:,}")
print(f"   High risk txns       : {high:,}")
print(f"   Models registered    : {len(model_names)}")

print("\n" + "=" * 55)
print("🎉 Pipeline 100% complete!")
print("   Ready for GitHub and LinkedIn!")
print("=" * 55)

   FRAUD DETECTION PIPELINE — FINAL CHECK

=== DELTA TABLES ===
✅ fraud_db.bronze_transactions                  284,807 rows
✅ fraud_db.silver_transactions                  281,918 rows
✅ fraud_db.gold_features                        281,918 rows
✅ fraud_db.gold_hourly_summary                  24 rows
✅ fraud_db.gold_predictions                     56,384 rows

=== MLFLOW REGISTERED MODELS ===
❌ fraud_RandomForest                  ERROR: INVALID_PARAMETER_VALUE: Bad model name: please specify all three levels of the model in the form `catalog_name.schema_name.model_name`
❌ fraud_XGBoost                       ERROR: INVALID_PARAMETER_VALUE: Bad model name: please specify all three levels of the model in the form `catalog_name.schema_name.model_name`
❌ fraud_LogisticRegression            ERROR: INVALID_PARAMETER_VALUE: Bad model name: please specify all three levels of the model in the form `catalog_name.schema_name.model_name`

=== PIPELINE SUMMARY ===
   Total transactions   : 281,918


In [0]:
# Find your exact catalog and schema
spark.sql("SELECT current_catalog(), current_schema()").show()

+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|        workspace|         default|
+-----------------+----------------+



In [0]:
# ============================================
# Fixed MLflow model check — Unity Catalog
# ============================================

import mlflow
client = mlflow.tracking.MlflowClient()

# Update these to match your catalog and schema
CATALOG = "workspace"
SCHEMA  = "personalproject(pp)"

model_names = [
    f"{CATALOG}.{SCHEMA}.fraud_RandomForest",
    f"{CATALOG}.{SCHEMA}.fraud_XGBoost",
    f"{CATALOG}.{SCHEMA}.fraud_LogisticRegression"
]

print("=== MLFLOW REGISTERED MODELS ===")
for name in model_names:
    try:
        versions = client.search_model_versions(
            f"name='{name}'"
        )
        for mv in versions:
            print(
                f"✅ {mv.name:50s} "
                f"v{mv.version}  "
                f"Stage: {mv.current_stage}"
            )
    except Exception as e:
        print(f"❌ {name} → ERROR: {e}")


=== MLFLOW REGISTERED MODELS ===
